In [ ]:
from google.colab import files
_ = files.upload()

Saving dataset-67110f40-705e-486c-b2ca-b0a635d92d32.zip to dataset-67110f40-705e-486c-b2ca-b0a635d92d32.zip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
# S1

def clasifica_greutate(carat):
    carat = round(carat, 4)
    if carat < 0.5:
        return 'Light'
    elif 0.5 <= carat < 1.5:
        return 'Medium'
    else:
        return 'Heavy'

test_df['weight'] = test_df['carat'].apply(clasifica_greutate)

# NOT -> test_df['weight'] = 'Light' if test_df[test_df['carat'] < 0.5] else 'Medium' if test_df[(test_df['carat'] >= 0.5) & (test_df['carat'] < 1.5)] else 'Heavy' if test_df[test_df['carat'] >= 1.5]

In [ ]:
"""Move column more to the front of the DataFrame"""
weight = test_df.pop('weight')

test_df.insert(2, 'weight', weight)

In [ ]:
test_df.head()

,SampleID,carat,cut,color,clarity,depth,table,x,y,z,weight
0,1389,0.24,Ideal,G,VVS1,62.1,56.0,3.97,4.00,2.47,Light
1,50053,0.58,Very Good,F,VVS2,60.0,57.0,5.44,5.42,3.26,Medium
2,41646,0.40,Ideal,E,VVS2,62.1,55.0,4.76,4.74,2.95,Light
3,42378,0.43,Premium,E,VVS2,60.8,57.0,4.92,4.89,2.98,Light
4,17245,1.55,Ideal,E,SI2,62.3,55.0,7.44,7.37,4.61,Heavy


In [ ]:
# S2

test_df['proportion'] = test_df['depth'] / test_df['table']

In [ ]:
"""Reformat new column position in the DataFrame"""
proportion = test_df.pop('proportion')

test_df.insert(8, 'proportion', proportion)

In [ ]:
test_df.head()

,SampleID,carat,cut,color,clarity,depth,table,x,y,z,weight,proportion
0,1389,0.24,Ideal,G,VVS1,62.1,56.0,3.97,4.00,2.47,Light,1.11
1,50053,0.58,Very Good,F,VVS2,60.0,57.0,5.44,5.42,3.26,Medium,1.05
2,41646,0.40,Ideal,E,VVS2,62.1,55.0,4.76,4.74,2.95,Light,1.13
3,42378,0.43,Premium,E,VVS2,60.8,57.0,4.92,4.89,2.98,Light,1.07
4,17245,1.55,Ideal,E,SI2,62.3,55.0,7.44,7.37,4.61,Heavy,1.13


In [ ]:
# S3

test_df['volume'] = test_df['x'] * test_df['y'] * test_df['z']

In [ ]:
test_df.head()

,SampleID,carat,cut,color,clarity,depth,table,x,y,z,weight,proportion,volume
0,1389,0.24,Ideal,G,VVS1,62.1,56.0,3.97,4.00,2.47,Light,1.11,39.22
1,50053,0.58,Very Good,F,VVS2,60.0,57.0,5.44,5.42,3.26,Medium,1.05,96.12
2,41646,0.40,Ideal,E,VVS2,62.1,55.0,4.76,4.74,2.95,Light,1.13,66.56
3,42378,0.43,Premium,E,VVS2,60.8,57.0,4.92,4.89,2.98,Light,1.07,71.70
4,17245,1.55,Ideal,E,SI2,62.3,55.0,7.44,7.37,4.61,Heavy,1.13,252.78


In [ ]:
train_df.head(3)

,SampleID,carat,cut,color,clarity,depth,table,price,x,y,z
0,19498,1.21,Ideal,H,VVS2,61.3,57.0,8131,6.92,6.87,4.23
1,31230,0.31,Ideal,E,VS2,62.0,56.0,756,4.38,4.36,2.71
2,22312,1.21,Ideal,E,VS1,62.4,57.0,10351,6.75,6.83,4.24


In [ ]:
np.unique(train_df['cut'])

array(['Fair', 'Good', 'Ideal', 'Premium', 'Very Good'], dtype=object)

In [ ]:
np.unique(train_df['color'])

array(['D', 'E', 'F', 'G', 'H', 'I', 'J'], dtype=object)

In [ ]:
np.unique(train_df['clarity'])

array(['I1', 'IF', 'SI1', 'SI2', 'VS1', 'VS2', 'VVS1', 'VVS2'],
      dtype=object)

In [ ]:
# S4 - Regression Task - Advanced Regresssion done with CatBoost (a bit overkill for this task but it received ~252 MAE)

"""Method 1 (worse)"""
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

X_train = train_df.drop(columns=['SampleID', 'price'])
y_train = train_df['price']

X_test = test_df.drop(columns=['SampleID'])

ordinal1 = ['Fair', 'Good', 'Ideal', 'Premium', 'Very Good']
ordinal2 = ['I1', 'IF', 'SI1', 'SI2', 'VS1', 'VS2', 'VVS1', 'VVS2']

onehot = [['D', 'E', 'F', 'G', 'H', 'I', 'J']]

trans = ColumnTransformer(
    transformers=[
        ('nominal', OneHotEncoder(drop='first', sparse_output=False), ['color']),
        ('ordinal', OrdinalEncoder(categories=[ordinal1, ordinal2]), ['cut', 'clarity'])
    ],
    remainder='passthrough'
)

X_train_trans = trans.fit_transform(X_train)
X_test_trans = trans.transform(X_test)

model = CatBoostRegressor(verbose=100, iterations=1500, depth=8, random_state=42, loss_function='MAE')

model.fit(X_train_trans, y_train)

0:	learn: 2756.7922811	total: 33.8ms	remaining: 50.7s
100:	learn: 491.6348479	total: 1.9s	remaining: 26.4s
200:	learn: 368.3075774	total: 3.32s	remaining: 21.5s
300:	learn: 327.6888697	total: 4.74s	remaining: 18.9s
400:	learn: 305.6125277	total: 6.19s	remaining: 17s
500:	learn: 293.1201981	total: 7.04s	remaining: 14s
600:	learn: 285.1956229	total: 7.87s	remaining: 11.8s
700:	learn: 279.2749415	total: 8.71s	remaining: 9.93s
800:	learn: 273.9078405	total: 9.57s	remaining: 8.35s
900:	learn: 269.3067715	total: 10.4s	remaining: 6.92s
1000:	learn: 265.2814835	total: 11.3s	remaining: 5.65s
1100:	learn: 261.7513225	total: 12.6s	remaining: 4.56s
1200:	learn: 259.1893269	total: 13.4s	remaining: 3.34s
1300:	learn: 256.1152557	total: 14.2s	remaining: 2.18s
1400:	learn: 252.8440730	total: 15.1s	remaining: 1.06s
1499:	learn: 250.0898412	total: 15.9s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1500, loss_function='MAE', random_state=42, verbose=100)

In [ ]:
"""Method 2 (better + simpler—didn't know that CatBoost can handle Ordinal categories without specific encoding)"""
X_train = train_df.drop(columns=['SampleID', 'price'])
y_train = train_df['price']

X_test = test_df.drop(columns=['SampleID', 'weight'])

cat_features = ['cut', 'color', 'clarity']

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, cat_features=cat_features)

model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function='MAE',
    eval_metric='MAE',
    random_state=42,
    verbose=100
)

model.fit(train_pool)

0:	learn: 2746.9945600	total: 75.9ms	remaining: 3m 47s
100:	learn: 460.5502583	total: 5.35s	remaining: 2m 33s
200:	learn: 322.6071510	total: 8.08s	remaining: 1m 52s
300:	learn: 294.2092574	total: 10.9s	remaining: 1m 38s
400:	learn: 281.7799667	total: 13.8s	remaining: 1m 29s
500:	learn: 273.2029815	total: 17.5s	remaining: 1m 27s
600:	learn: 266.8826915	total: 22.5s	remaining: 1m 29s
700:	learn: 262.7368910	total: 28.8s	remaining: 1m 34s
800:	learn: 258.3844010	total: 34.3s	remaining: 1m 34s
900:	learn: 254.7466552	total: 37.3s	remaining: 1m 26s
1000:	learn: 251.5468014	total: 40.9s	remaining: 1m 21s
1100:	learn: 248.1511535	total: 43.9s	remaining: 1m 15s
1200:	learn: 245.6990521	total: 46.9s	remaining: 1m 10s
1300:	learn: 243.4513678	total: 50s	remaining: 1m 5s
1400:	learn: 241.0674558	total: 53.7s	remaining: 1m 1s
1500:	learn: 239.1438334	total: 56.6s	remaining: 56.5s
1600:	learn: 237.1451516	total: 59.5s	remaining: 52s
1700:	learn: 235.4459279	total: 1m 2s	remaining: 48s
1800:	learn: 

CatBoostRegressor(depth=8, eval_metric='MAE', iterations=3000, learning_rate=0.03, loss_function='MAE', random_state=42, verbose=100)

In [ ]:
y_preds = model.predict(X_test)

y_preds

array([  547.74907274,  2283.87189834,  1224.12925625, ...,
       12839.80014802,  3237.89578126,  1318.60027808])

In [ ]:
"""Internal performance interpretation"""
from sklearn.metrics import mean_absolute_error as mae

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2)

y_pred = model.predict(X_val)

mae_val = mae(y_val, y_pred)

mae_val

218.11471543866114

In [ ]:
"""Submission DataFrame building + downloading"""
DF1 = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': test_df['SampleID'],
    'answer': test_df['weight'],
})

#DF1 (uncomment for printing each subset of the submission)

In [ ]:
DF2 = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': test_df['SampleID'],
    'answer': test_df['proportion'],
})

#DF2 (uncomment for printing each subset of the submission)

In [ ]:
DF3 = pd.DataFrame({
    'subtaskID': 3,
    'datapointID': test_df['SampleID'],
    'answer': test_df['volume'],
})

#DF3 (uncomment for printing each subset of the submission)

In [ ]:
DF4 = pd.DataFrame({
    'subtaskID': 4,
    'datapointID': test_df['SampleID'],
    'answer': y_preds,
})

#DF4 (uncomment for printing each subset of the submission)

In [ ]:
submission = pd.concat([DF1, DF2, DF3, DF4])
submission.to_csv('submission.csv', index=False)

In [ ]:
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>